# IT Support Dashboard - Supporting Notebook 1

This series of notebooks contains supporting documentation to the full report. Here, we provide quality checks for the full dataset, confirm whether the English-only sample is representitive of the full population available, and conduct text analytics.

## Cleaning Process
Notebook 1 provides the data cleaning process of the raw dataset. Records containing null values within the Answers column where removed, tags were tidied, and variations in data entry of Priority and Language columnns were standardised.


### Importing
First, the required packages for this process were imported, the necessary folders were created using the config file, and the raw dataset was read.

In [12]:
# Importing essential packages
import sys
from pathlib import Path
import pandas as pd

# Add parent directory (project root) to sys.path
project_root = Path(r"C:\Users\David\Desktop\Python_Files\IT-Support-Ticket-Analysis")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import Config

Config.ensure_directories()

# Reading the initial dataset as a DataFrame: df
df = pd.read_csv(
    Config.RAW_DATA_PATH,
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip",
    na_values=[
        "",
        " ",
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "None",
        "none",
        "NAN",
        "NaN",
        "nan",
        None,
    ],
)

df  # Display the DataFrame to verify successful import

[Config] Verified project directory structure under C:\Users\David\Desktop\Python_Files\IT-Support-Ticket-Analysis


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28582,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
28583,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
28584,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
28585,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


Second, we wanted to see how clean and complete this dataset was. We were looking at the shape of the table, any missing values, if dimensions were in appropriate datatypes, and how much memory they used.

In [13]:
# Information about the dataset
print("What is the shape of my table?")
print(df.shape)
print("\nAre there any missing values in each dimension?")
print(df.isna().sum().sort_values())
print("\nWhat is the datatype of each column?")
print(df.dtypes)
print("\nHow many bytes does each column use?")
print(df.memory_usage())

What is the shape of my table?
(28587, 16)

Are there any missing values in each dimension?
body            0
type            0
queue           0
priority        0
language        0
version         0
tag_1           0
answer          7
tag_2          13
tag_3         136
tag_4        3058
subject      3838
tag_5       14042
tag_6       22713
tag_7       26547
tag_8       28022
dtype: int64

What is the datatype of each column?
subject     object
body        object
answer      object
type        object
queue       object
priority    object
language    object
version      int64
tag_1       object
tag_2       object
tag_3       object
tag_4       object
tag_5       object
tag_6       object
tag_7       object
tag_8       object
dtype: object

How many bytes does each column use?
Index          132
subject     228696
body        228696
answer      228696
type        228696
queue       228696
priority    228696
language    228696
version     228696
tag_1       228696
tag_2       228696
tag_

### Removing null responses

Seven null entries were identified and removed from the Answers column. We removed them because we thought these would intefere with the text analytics planned later on, and were comfortable with losing 0.02% of total records. This was the simplest solution, as it wasn't possible to reconstruct the original answer provided by the Customer Service representative. 

In [14]:
# Identify rows with null answers; subjects and tags were ignored as they were less critical to this analysis
null_answers = df[df["answer"].isna()]

# Count the number of null answers
null_count = null_answers.shape[0]

# Perecnentage of null answers
percentage_null = (null_count / df.shape[0]) * 100

print(
    f"\nTable containing {null_count} rows ({percentage_null:.2f}%) with null answers:"
)
display(null_answers)


Table containing 7 rows (0.02%) with null answers:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
4956,Support Inquiry Regarding SendGrid Integration,"Dear Customer Service, I would like to inquire...",NaN,Request,Billing and Payments,high,de,52,Billing,Payment,Platform,Pricing,Discount,Promotion,Integration,Support
5381,Support Request for SendGrid Integration,"Dear Customer Service, I would like to inquire...",NaN,Request,Billing and Payments,high,de,52,Billing,Payment,Support,Integration,Pricing,Promotion,NaN,NaN
12163,Recent Decline in Engagement Metrics Noted Online,We have observed a decrease in engagement metr...,NaN,Problem,Product Support,medium,en,400,Feedback,Performance,Feature,NaN,NaN,NaN,NaN,NaN
13378,Recent Decrease in Engagement Metrics Noted On...,There has been a decline in engagement metrics...,NaN,Problem,Product Support,medium,en,400,Feedback,Performance,Feature,NaN,NaN,NaN,NaN,NaN
13651,NaN,We are sorry to hear that you are experiencing...,NaN,Problem,Technical Support,high,en,400,Bug,Performance,Feature,Documentation,Tech Support,NaN,NaN,NaN
16596,"Sicher, Benutzer melden zeitweise Verbindungsp...","rufen Sie uns an <tel_num>, um über weitere Lö...",NaN,Incident,Customer Service,low,de,400,Network,Performance,Disruption,IT,Tech Support,NaN,NaN,NaN
26522,Reduction in Engagement Metrics Noted Online,Observation of a decline in engagement metrics...,NaN,Problem,Product Support,medium,de,400,Feedback,Performance,Feature,Documentation,NaN,NaN,NaN,NaN


In [15]:
# Dropping null answers and resetting index
df = df.dropna(subset=["answer"])
df.reset_index(drop=True, inplace=True)

print("\nNumber of null answers after dropping:", df["answer"].isna().sum())


Number of null answers after dropping: 0


### Amending tags

Next, we corrected anomalies within the tags. Under the tag_1 column, 12 records were found with various tags stitched together using a comma. Seeing as there were no entries within the other tag columns, we assumed these were entered erroneously. These sequences of tags were separated and used to populate the other tag columns for each record.

Under the tag_3 column, 1 record was identified with a similar validation error. These tags were separated and used to populate succeeding field, shuffling the tags downward and maintaining the order of input.

Similar issues relating to tags were not found within the other columns.

In [16]:
# Number of tag columns
tag_columns = [col for col in df.columns if col.startswith("tag_")]
print(f"\nNumber of tag columns: {len(tag_columns)}")

long_tags_columns = []

# Loop to chechk for multiple tags
for i in range(1, len(tag_columns) + 1):
    tag_col = f"tag_{i}"
    if tag_col in df.columns:
        long_tags = df[df[tag_col].str.contains(",", na=False)]

        if long_tags.empty:
            print(f"\nNo rows with multiple tags found in {tag_col}.")
        else:
            long_tags_columns.append(tag_col)
            long_tags_count = long_tags.shape[0]
            print(
                f"\nTable containing {long_tags_count} rows with multiple tags in {tag_col}:"
            )
            display(long_tags)


Number of tag columns: 8

Table containing 12 rows with multiple tags in tag_1:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
958,Digitale Kampagnen erzielten unterdurchschnitt...,Die digitalen Kampagnen der Agentur schnitten ...,Überprüfen Sie die digitalen Kampagnen und ide...,Problem,Product Support,low,de,52,"Performance,Bug,Disruption,Security",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1784,Significant Underperformance in Today's Digita...,The marketing agency's digital advertising eff...,We received an email concerning the underperfo...,Problem,Technical Support,low,en,52,"Performance,Disruption,Outage,Monitoring,Analysis",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2166,Heute ist die Investmentplattform abgestürzt,"Sehr geehrter Kundendienst, ich möchte ein Pro...","<name>, vielen Dank, dass Sie das Problem unse...",Incident,Technical Support,high,de,52,"Crash,Performance,Outage,Disruption,Recovery,S...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2331,Significant Underperformance of Digital Campai...,The digital campaigns managed by the marketing...,We received your email regarding the underperf...,Incident,Technical Support,high,en,52,"Performance,Disruption,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2410,Digitale Kampagnen erzielen plattformübergreif...,Die digitalen Kampagnen unserer Marketingagent...,"Die digitalen Kampagnen überprüfen, Telefonnum...",Problem,Product Support,high,de,52,"Performance,Disruption,Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3146,Digitale Marketingkampagne der Agentur zeigt h...,Die digitale Kampagne der Marketingagentur wei...,Ich habe eine E-Mail bezüglich der schlechten ...,Problem,Technical Support,low,de,52,"Performance,Outage,Disruption,Recovery,Marketi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3537,Recent Unauthorized Access Attempts Detected o...,There have been recent attempts to gain unauth...,We received an email concerning unauthorized a...,Incident,Technical Support,medium,en,52,"Security,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4758,Digitale Kampagnen schneiden plattformübergrei...,"Sehr geehrter Kundenservice, ich wende mich be...","<name>, ich helfe Ihnen gern bei der Fehlerbeh...",Problem,Technical Support,high,de,52,"Performance,Disruption,Outage,Support,Integration",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5354,Fortschrittliche Datenanalyse,Bitte nutzen Sie detaillierte Informationen zu...,"<name>, vielen Dank für Ihre Anfrage bezüglich...",Request,Technical Support,high,de,52,"Performance,Security,Feature,Documentation",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5696,Hospital System Security Practices,Customer Support is inquiring about methods to...,"<name>, we understand your concerns regarding ...",Request,IT Support,high,en,52,"Security,IT,Tech Support,Data Privacy,Regulati...",NaN,NaN,NaN,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_2.

Table containing 1 rows with multiple tags in tag_3:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
5883,Zugang zu medizinischen Daten,"Sehr geehrter Kundenservice, ich möchte Sie da...","Sehr geehrter <name>, wir nehmen die Sicherhei...",Problem,Technical Support,medium,de,52,Security,Network,"Disruption,IT",Tech Support,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_4.

No rows with multiple tags found in tag_5.

No rows with multiple tags found in tag_6.

No rows with multiple tags found in tag_7.

No rows with multiple tags found in tag_8.


In [17]:
print(f"Columns {long_tags_columns} contain multiple tags separated by commas.")

Columns ['tag_1', 'tag_3'] contain multiple tags separated by commas.


In [18]:
# Creating a loop to populate the other tag columns with the additional tags from tag_1, then leaving the original tag_1 column with the first tag only
for col in long_tags_columns:
    if col == "tag_1":
        # Store the indices of rows with multiple tags in tag_1
        index_list = df[df[col].str.contains(",", na=False)].index.tolist()
        for i in index_list:
            row = df.loc[i]
            print(f"Row {i}: {row['tag_1']}")
            tags = row["tag_1"].split(",")
            # Update tag_2, tag_3, tag_4 with additional tags
            df.loc[row.name, "tag_2"] = tags[1].strip() if len(tags) > 1 else None
            df.loc[row.name, "tag_3"] = tags[2].strip() if len(tags) > 2 else None
            df.loc[row.name, "tag_4"] = tags[3].strip() if len(tags) > 3 else None
            # Update tag_1 to only contain the first tag
            df.loc[row.name, "tag_1"] = tags[0].strip()

    if col == "tag_3":
        # Store the indices of rows with multiple tags in tag_3
        index_list.extend(df[df[col].str.contains(",", na=False)].index.tolist())
        for i in df[df[col].str.contains(",", na=False)].index.tolist():
            row = df.loc[i]
            print(f"Row {i}: {row['tag_3']}")
            tags = row["tag_3"].split(",")
            # Move tag_4 to tag_5, then update tag_4 with the second tag from tag_3
            df.loc[row.name, "tag_5"] = row["tag_4"]
            df.loc[row.name, "tag_4"] = tags[1].strip() if len(tags) > 1 else None
            # Update tag_3 to only contain the first tag
            df.loc[row.name, "tag_3"] = tags[0].strip()

# Checking the affected records to verify whether the changes were successfully implemented
print("\nUpdated rows with multiple tags:")
display(df[df.index.isin(index_list)])

Row 958: Performance,Bug,Disruption,Security
Row 1784: Performance,Disruption,Outage,Monitoring,Analysis
Row 2166: Crash,Performance,Outage,Disruption,Recovery,Server,DataProcessing
Row 2331: Performance,Disruption,IT,Tech Support
Row 2410: Performance,Disruption,Support
Row 3146: Performance,Outage,Disruption,Recovery,Marketing,Agentur,Analyse
Row 3537: Security,IT,Tech Support
Row 4758: Performance,Disruption,Outage,Support,Integration
Row 5354: Performance,Security,Feature,Documentation
Row 5696: Security,IT,Tech Support,Data Privacy,Regulation,Patient Data,Threat Prevention
Row 7287: Security,Outage,Disruption,Recovery,IT,Tech Support
Row 7472: Performance,Feature,Documentation,Feedback
Row 5883: Disruption,IT

Updated rows with multiple tags:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
958,Digitale Kampagnen erzielten unterdurchschnitt...,Die digitalen Kampagnen der Agentur schnitten ...,Überprüfen Sie die digitalen Kampagnen und ide...,Problem,Product Support,low,de,52,Performance,Bug,Disruption,Security,NaN,NaN,NaN,NaN
1784,Significant Underperformance in Today's Digita...,The marketing agency's digital advertising eff...,We received an email concerning the underperfo...,Problem,Technical Support,low,en,52,Performance,Disruption,Outage,Monitoring,NaN,NaN,NaN,NaN
2166,Heute ist die Investmentplattform abgestürzt,"Sehr geehrter Kundendienst, ich möchte ein Pro...","<name>, vielen Dank, dass Sie das Problem unse...",Incident,Technical Support,high,de,52,Crash,Performance,Outage,Disruption,NaN,NaN,NaN,NaN
2331,Significant Underperformance of Digital Campai...,The digital campaigns managed by the marketing...,We received your email regarding the underperf...,Incident,Technical Support,high,en,52,Performance,Disruption,IT,Tech Support,NaN,NaN,NaN,NaN
2410,Digitale Kampagnen erzielen plattformübergreif...,Die digitalen Kampagnen unserer Marketingagent...,"Die digitalen Kampagnen überprüfen, Telefonnum...",Problem,Product Support,high,de,52,Performance,Disruption,Support,None,NaN,NaN,NaN,NaN
3146,Digitale Marketingkampagne der Agentur zeigt h...,Die digitale Kampagne der Marketingagentur wei...,Ich habe eine E-Mail bezüglich der schlechten ...,Problem,Technical Support,low,de,52,Performance,Outage,Disruption,Recovery,NaN,NaN,NaN,NaN
3537,Recent Unauthorized Access Attempts Detected o...,There have been recent attempts to gain unauth...,We received an email concerning unauthorized a...,Incident,Technical Support,medium,en,52,Security,IT,Tech Support,None,NaN,NaN,NaN,NaN
4758,Digitale Kampagnen schneiden plattformübergrei...,"Sehr geehrter Kundenservice, ich wende mich be...","<name>, ich helfe Ihnen gern bei der Fehlerbeh...",Problem,Technical Support,high,de,52,Performance,Disruption,Outage,Support,NaN,NaN,NaN,NaN
5354,Fortschrittliche Datenanalyse,Bitte nutzen Sie detaillierte Informationen zu...,"<name>, vielen Dank für Ihre Anfrage bezüglich...",Request,Technical Support,high,de,52,Performance,Security,Feature,Documentation,NaN,NaN,NaN,NaN
5696,Hospital System Security Practices,Customer Support is inquiring about methods to...,"<name>, we understand your concerns regarding ...",Request,IT Support,high,en,52,Security,IT,Tech Support,Data Privacy,NaN,NaN,NaN,NaN


### Standardising inputs

Then we tidied up the dataset by removing variations within the Priority and Language columns, as well as changed the datatypes to reduce memory usage, improving performance.

In [19]:
# Standardizing text in 'priority' and 'language' columns to title case
df["priority"] = df["priority"].str.title()
df["language"] = df["language"].str.upper()

# Converting data types to save on memory usage
# Lists of data types by column
categories = [
    "type",
    "queue",
    "priority",
    "language",
    "version",
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4",
    "tag_5",
    "tag_6",
    "tag_7",
    "tag_8",
]
strings = ["subject", "body", "answer"]

# List comprehension loop
for types in df:
    # Categories
    for category in categories:
        if types == category:
            df[types] = df[types].astype("category")

    # Integers
    for string in strings:
        if types == string:
            df[types] = df[types].astype("string")

# Displaying the DataFrame after dropping null answers, resetting index, and converting data types
print("\nDataframe after dropping null answers and resetting index:")
display(df)


Dataframe after dropping null answers and resetting index:


C:\Users\David\AppData\Local\Temp\ipykernel_388\3205797046.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["priority"] = df["priority"].str.title()
C:\Users\David\AppData\Local\Temp\ipykernel_388\3205797046.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["language"] = df["language"].str.upper()
C:\Users\David\AppData\Local\Temp\ipykernel_388\3205797046.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = va

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,High,DE,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,High,EN,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,Medium,EN,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,Low,EN,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,Medium,EN,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28575,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,High,EN,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
28576,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,High,DE,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
28577,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,Low,DE,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
28578,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,High,EN,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


In [20]:
print("What is the shape of my table?")
print(df.shape)
print("\nAre there any missing values in each dimension?")
print(df.isna().sum().sort_values())
print("\nWhat is the datatype of each column?")
print(df.dtypes)
print("\nHow many bytes does each column use?")
print(df.memory_usage())

What is the shape of my table?
(28580, 16)

Are there any missing values in each dimension?
body            0
answer          0
type            0
queue           0
priority        0
language        0
version         0
tag_1           0
tag_2           1
tag_3         124
tag_4        3046
subject      3837
tag_5       14038
tag_6       22708
tag_7       26541
tag_8       28016
dtype: int64

What is the datatype of each column?
subject     string[python]
body        string[python]
answer      string[python]
type              category
queue             category
priority          category
language          category
version           category
tag_1             category
tag_2             category
tag_3             category
tag_4             category
tag_5             category
tag_6             category
tag_7             category
tag_8             category
dtype: object

How many bytes does each column use?
Index          132
subject     228640
body        228640
answer      228640
type     

Continuing on with the quality checks, we wanted to confirm how many unique values there were under each dimension.

In [21]:
# Loop to extract all unique values from each column in df
for column in df.columns:
    unique_values = df[column].sort_values(ascending=True).unique()
    length = len(unique_values)
    print(f"There were {length} unique values in {column}:\n{unique_values}")
    print("")

There were 24744 unique values in subject:
<StringArray>
[                                                                                                                                                                                              ' Assistance Request',
                                                                                                                                         ' Bitte um Ausführliche Informationen zur Datenaufbereitungsdienstleistung',
                                                                                                                                                                   ' Datenschutzverletzung in Krankenhaus-Systemen ',
                                                                                                                                                                                ' Reported Problem with Data Access',
                                                                                       

### Saving cleaned dataset

Two versions were saved, CSV and Parquet, for downstream use.

In [22]:
# Importing os package for file operations
import os

# Define file names
csv_path = Config.QC_OUTPUT_DIR / "Tickets_Clean.csv"
parquet_path = Config.QC_OUTPUT_DIR / "Tickets_Clean.parquet"

# CSV – portable
df.to_csv(csv_path, index=False, encoding="utf-8")

# Parquet – efficient
df.to_parquet(parquet_path, index=False, engine="pyarrow", compression="snappy")

# Summary of saved files
for path in [csv_path, parquet_path]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{path.name:<25} | {size_mb:>6.2f} MB")

print("\nAll files saved successfully.")

Tickets_Clean.csv         |  24.79 MB
Tickets_Clean.parquet     |  11.30 MB

All files saved successfully.
